In [2]:
# TODO: Nothing I tried works. Just restart the notebook kernel when you change anything.

import sys
from pathlib import Path

# Ensure the notebook imports the local src tree, not an installed copy.
src_dir = Path.cwd() / "src"
if str(src_dir) in sys.path:
    sys.path.remove(str(src_dir))
sys.path.insert(0, str(src_dir))


# Reload to pick up latest changes
from importlib import reload

import data_labeling_pipeline as dlp

reload(dlp)
reload(dlp.hyperparameters)


# # Use the %load_ext autoreload magic command to enable automatic reloading.
# %load_ext autoreload
# # Use %autoreload 2 to specify that all modules should be reloaded before executing the Python code.
# %autoreload 2

<module 'data_labeling_pipeline.hyperparameters' from '/workspaces/master-thesis-lmertens/data_labeling_pipeline/src/data_labeling_pipeline/hyperparameters.py'>

In [ ]:
import numpy as np
from config.paths import EXTRACTED_ROOT, LABELS_ROOT, ROS1_ROOT
from data_labeling_pipeline.grandtour_missions import (
    get_mission_by_shortname,
)

mission_shortname = "ETH-1"
mission = get_mission_by_shortname(mission_shortname)

mission_path = ROS1_ROOT / mission_shortname

assert mission_path.exists(), f"Mission path {mission_path} does not exist"

extraction_dir = EXTRACTED_ROOT / mission_shortname
labels_dir = LABELS_ROOT / mission_shortname / "rhe"
labels_dir.mkdir(parents=True, exist_ok=True)


In [6]:
# import numpy as np
# from data_labeling_pipeline.dataset_loader import GrandTourLoader

# dataset_loader = GrandTourLoader(mission_path=mission_path)
# cmd_ts, odom_ts = dataset_loader.load_data_shared_timestamps()


# print(np.where(np.diff(cmd_ts) < 0)[0])

# # Find the timestamps where the command timestamps are not strictly increasing, which indicates a reset in the data collection.
# mask = np.diff(cmd_ts, append=np.inf) < 0
# indexes = np.where(mask)[0]

# for idx in indexes:
#     print(
#         f"Non-increasing timestamp at index {idx}: {cmd_ts[idx - 1]} -> {cmd_ts[idx]} --> {cmd_ts[idx + 1]}"
#     )


In [ ]:
from data_labeling_pipeline import DataLabelingPipeline

pipeline = DataLabelingPipeline(
    mission_path=mission_path, extraction_dir=extraction_dir, labels_dir=labels_dir
)


pipeline.process_mission()

In [ ]:
import matplotlib.pyplot as plt
from data_labeling_pipeline.dataset_loader import GrandTourLoader

# reload(GrandTourLoader)

loader = GrandTourLoader(mission_path)
shared_ts, U, Z = loader.load_data_shared_timestamps(dt=0.1, fill_gaps=True)


v_cmd, omega_cmd = U.T
px, py, yaw = Z.T
vx = np.gradient(px, shared_ts)
vy = np.gradient(py, shared_ts)
v_forward = np.cos(yaw) * vx + np.sin(yaw) * vy
omega_act = np.gradient(np.unwrap(yaw), shared_ts)


# Scalar Kalman filter for a noisy 1-D error signal.
def kalman_smooth(signal, process_variance=1e-4, measurement_variance=1e-2):
    filtered = np.empty_like(signal, dtype=float)
    estimate = float(signal[0])
    covariance = 1.0

    for index, measurement in enumerate(signal):
        covariance = covariance + process_variance
        kalman_gain = covariance / (covariance + measurement_variance)
        estimate = estimate + kalman_gain * (measurement - estimate)
        covariance = (1.0 - kalman_gain) * covariance
        filtered[index] = estimate

    return filtered


cmd_ts_prime, U_prime, Z_prime = loader.load_data_shared_timestamps(
    dt=0.1, fill_gaps=True, include_lateral_command=True
)

# Kalman-smoothed absolute errors.
dv_raw = np.square(v_cmd - v_forward)
d_omega_raw = np.square(omega_cmd - omega_act)
dv = kalman_smooth(dv_raw)
d_omega = kalman_smooth(d_omega_raw)


# Map the filtered error through a sigmoid to obtain a bounded traversability score.
# Define the thresholds and steepness directly in the notebook.

v_thresh = 0.2  # err^2 lower than 0.4 --> traversability > 0.5
w_thresh = 0.2
k_v = 5.0 / v_thresh
k_w = 5.0 / w_thresh


def traversability_score(error_signal, threshold, steepness):
    return 1.0 / (1.0 + np.exp(steepness * (error_signal - threshold)))


tau_v = traversability_score(dv, v_thresh, k_v)
tau_w = traversability_score(d_omega, w_thresh, k_w)


# Load traversability labels and align to shared timeline
label_ts = np.load(labels_dir / "rhe_timestamp.npy")
trav_labels = np.load(labels_dir / "rhe_trav_labels.npy")
nu_confidences = np.load(labels_dir / "rhe_nu_observability.npy")

mu_label = trav_labels[:, 0]
nu_label = trav_labels[:, 1]
delta_theta_label = trav_labels[:, 2]


mu = np.interp(shared_ts, label_ts, mu_label)
nu = np.interp(shared_ts, label_ts, nu_label)
delta_theta = np.interp(shared_ts, label_ts, delta_theta_label)
nu_confidence = np.interp(shared_ts, label_ts, nu_confidences)


# Try to load mu_confidence
try:
    mu_confidences = np.load(labels_dir / "rhe_mu_observability.npy")
    mu_confidence = np.interp(shared_ts, label_ts, mu_confidences)
except FileNotFoundError:
    # If not available, use all ones (perfect confidence)
    mu_confidence = np.ones_like(shared_ts)

fig, axes = plt.subplots(8, 1, figsize=(13, 15), sharex=True)

axes[0].plot(shared_ts, v_cmd, label="v_cmd")
axes[0].plot(shared_ts, v_forward, label="v_forward (body x)", alpha=0.85)
axes[0].set_ylabel("Linear [m/s]")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(shared_ts, dv_raw, label="|v_cmd - v_forward|", alpha=0.35)
axes[1].plot(shared_ts, dv, label="Kalman-smoothed v_error")
axes[1].plot(shared_ts, 1 - mu, label="1-mu", color="tab:green")
axes[1].set_ylabel("Error vs. traversability label (linear)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(shared_ts, omega_cmd, label="omega_cmd")
axes[2].plot(shared_ts, omega_act, label="omega_act = d(yaw)/dt", alpha=0.5)
axes[2].set_ylabel("Angular [rad/s]")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

axes[3].plot(shared_ts, d_omega_raw, label="|omega_cmd - omega_act|", alpha=0.35)
axes[3].plot(shared_ts, d_omega, label="Kalman-smoothed w_error")
axes[3].plot(shared_ts, 1 - nu, label="1-nu", color="tab:orange")
axes[3].set_ylabel("Error vs. traversability label (angular)")
axes[3].set_xlabel("Time [s]")
axes[3].legend()
axes[3].grid(True, alpha=0.3)

axes[4].plot(shared_ts, nu_confidence, label="nu_confidence", color="tab:purple")
axes[4].set_ylabel("Angular Confidence")
axes[4].set_xlabel("Time [s]")
axes[4].legend()
axes[4].grid(True, alpha=0.3)

axes[5].plot(shared_ts, delta_theta, label="delta_theta")
axes[5].set_ylabel("Heading error label [rad]")
axes[5].set_xlabel("Time [s]")
axes[5].legend()
axes[5].grid(True, alpha=0.3)

# axes[5].plot(shared_ts, yaw, label="yaw")
# axes[5].set_ylabel("Yaw [rad]")
# axes[5].set_xlabel("Time [s]")
# axes[5].legend()

# Plot the pearson correcelation with a rolling window
window_size = shared_ts.shape[0] - 1
window_size = 100


rolling_corr_v = np.array([
    np.corrcoef(dv[i : i + window_size], 1 - mu[i : i + window_size])[0, 1]
    for i in range(len(dv) - window_size)
])
rolling_corr_omega = np.array([
    np.corrcoef(d_omega[i : i + window_size], 1 - nu[i : i + window_size])[0, 1]
    for i in range(len(d_omega) - window_size)
])

print(rolling_corr_v)
print(rolling_corr_omega)
axes[6].plot(
    shared_ts[: len(rolling_corr_v)],
    rolling_corr_v,
    label="Rolling Corr (Kalman v_error vs 1-mu)",
)
axes[6].plot(
    shared_ts[: len(rolling_corr_omega)],
    rolling_corr_omega,
    label="Rolling Corr (Kalman w_error vs 1-nu)",
)
axes[6].set_ylabel("Rolling Pearson Correlation")
axes[6].set_xlabel("Time [s]")
axes[6].legend()
axes[6].grid(True, alpha=0.3)

axes[7].plot(shared_ts, mu, label="mu (linear traversability)", color="tab:blue")
axes[7].plot(
    shared_ts,
    tau_v,
    label="tau_v = sigmoid(-k(v_error-v_thr))",
    color="tab:blue",
    linestyle="--",
)
axes[7].plot(shared_ts, nu, label="nu (angular traversability)", color="tab:red")
axes[7].plot(
    shared_ts,
    tau_w,
    label="tau_w = sigmoid(-k(w_error-w_thr))",
    color="tab:red",
    linestyle="--",
)
axes[7].set_ylabel("Traversability")
axes[7].set_xlabel("Time [s]")
axes[7].legend()
axes[7].grid(True, alpha=0.3)


# Add gradient shading based on confidence values (0=red, 0.5=orange, 1=green)
def add_confidence_shading(ax, time, confidence, cmap_name="RdYlGn"):
    """Add thin vertical lines colored by confidence values (0=red, 0.5=orange, 1=green)"""
    import matplotlib.cm as cm
    from matplotlib.collections import LineCollection
    from matplotlib.colors import Normalize

    # Get current y limits
    ylim = ax.get_ylim()

    # Prepare line segments
    segments = []
    colors_list = []

    for i in range(len(time)):
        segments.append([(time[i], ylim[0]), (time[i], ylim[1])])
        colors_list.append(confidence[i])

    # Create colormap
    cmap = cm.get_cmap(cmap_name)
    norm = Normalize(vmin=0, vmax=1)

    # Create line collection with very thin lines
    lc = LineCollection(
        segments, cmap=cmap, norm=norm, linewidths=0.05, zorder=-1, alpha=0.5
    )
    lc.set_array(np.array(colors_list))

    ax.add_collection(lc)
    ax.set_ylim(ylim)


# Add red shading where commands are near zero
def add_zero_shading(ax, time, condition, color="red", alpha=0.2):
    diff = np.diff(condition.astype(int))
    starts = np.where(diff == 1)[0] + 1
    ends = np.where(diff == -1)[0]
    if condition[0]:
        starts = np.concatenate([[0], starts])
    if condition[-1]:
        ends = np.concatenate([ends, [len(condition) - 1]])
    for s, e in zip(starts, ends):
        ax.axvspan(
            time[s],
            time[e],
            color=color,
            alpha=alpha,
        )


# Add confidence shading
add_confidence_shading(axes[1], shared_ts, mu_confidence, cmap_name="RdYlGn")
add_confidence_shading(axes[3], shared_ts, nu_confidence, cmap_name="RdYlGn")
add_confidence_shading(axes[7], shared_ts, nu_confidence, cmap_name="RdYlGn")


# Add zero command shading
# add_zero_shading(axes[1], shared_ts, np.isclose(v_cmd, 0))
# add_zero_shading(axes[3], shared_ts, np.abs(omega_cmd) > 0.001, color="green")

fig.tight_layout()
plt.show()
